In [1]:
import os
os.chdir('/home/asudupe/Latxa-Omni/dataset_generation/vits')

In [2]:
os.getcwd()

'/home/asudupe/Latxa-Omni/dataset_generation/vits'

In [3]:
import pyximport
pyximport.install()

(None, <pyximport.pyximport.PyxImportMetaFinder at 0x14acd8b780a0>)

In [26]:
from datasets import load_dataset
dataset = load_dataset("Ansu/Instruct_200k_eu", split="train")

In [27]:
dataset[186148]

{'id': 'instruct_eu_186148',
 'conversation': [{'from': 'human',
   'speech': 'instruct_eu_186148_user_0.wav',
   'text': 'Beraz, zer joko periferikoak kontrolatzaileak eta gauzak, benetan Xbox aplikazioa bateragarriak dira?',
   'unit': None},
  {'from': 'gpt',
   'speech': 'instruct_eu_186148_assistant_0.wav',
   'text': 'Xbox aplikazioak Xbox kontroladoreak onartzen ditu, Xbox Elite Series Two, Xbox Adaptive Controller eta Xbox Wireless Controller barne.Gainera, hirugarrenen kontrolagailu batzuk, Razer Raiju eta PowerA kontroladoreak bezala, bateragarriak dira.',
   'unit': None}]}

In [22]:
dataset.push_to_hub("Ansu/Instruct_200k_eu_filtered_8_aaa", private=True)

Creating parquet from Arrow format: 100%|██████████| 331/331 [00:02<00:00, 153.43ba/s]
Processing Files (1 / 1): 100%|██████████| 61.2MB / 61.2MB, 6.12MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:28<00:00, 28.91s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/Ansu/Instruct_200k_eu_filtered_8_aaa/commit/24263d15da9cc00a3463f4c1cbccdebb9ade23da', commit_message='Upload dataset', commit_description='', oid='24263d15da9cc00a3463f4c1cbccdebb9ade23da', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Ansu/Instruct_200k_eu_filtered_8_aaa', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Ansu/Instruct_200k_eu_filtered_8_aaa'), pr_revision=None, pr_num=None)

In [4]:
text = dataset[527]['answer']
cleaned_text = text.replace(':', '')
print(cleaned_text)
command = f"echo {cleaned_text} | iconv -f UTF-8 -t ISO-8859-1 | ./modulo1y2 -HDic=dict/eu_dic -Lang=eu -TxtMode=Spell -PhTSimple=y 2> /dev/null | iconv -f ISO-8859-1 -t UTF-8"
phones = os.popen(command).read()

Hona hemen horietako bat “Jantzi zure konfiantza” (Jantzi zure konfiantza)


/bin/sh: 1: Syntax error: "(" unexpected


In [216]:
phones

"S i - J 'i n | d i s - t 'i - r a - ts u - a - G o - a | e - G 'u s` - k i | e - n 'e rr - g i - a k | k a rr - g 'a | s` u - s` 'e n - ts` e n | d 'u | a - m 'e - r i - k a - k o | e - t 'o rr - k i - s` u n | b e - rr 'i s` - t a - G a - rr i - a n \n"

In [4]:
# import matplotlib.pyplot as plt
# import IPython.display as ipd

import os
# import json
# import math
import torch
# from torch import nn
# from torch.nn import functional as F
# from torch.utils.data import DataLoader
import IPython.display as ipd
import commons
from utils import get_hparams_from_file, load_checkpoint
# from data_utils import TextAudioLoader, TextAudioCollate, TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text.symbols_cast import symbols_cast
from text import text_to_sequence

from scipy.io.wavfile import write

import soundfile
import speech
import numpy as np
import time
# import sys

# Verifica qué speech se está usando (LIBBERTSO o /vits)
print("Usando speech desde:", speech.__file__)

def clean_text(text):
    text = text.replace(':', ',')
    text = text.replace(';', ',')
    text = text.replace('(', ',')
    text = text.replace(')', ',')
    text = text.replace('"', '')
    text = text.replace("'", '')
    text = text.replace("“", '')
    text = text.replace("”", '')
    text = text.replace("ñ", 'n')
    text = text.replace("á", 'a')
    text = text.replace("é", 'e')
    text = text.replace("í", 'i')
    text = text.replace("ó", 'o')
    text = text.replace("ú", 'u')
    # print(output)
    return text

def getPhones(text, language):
    #####################################
    # Extracción fonética de las frases #
    #####################################
    text = text.lstrip()
    cleaned_text = clean_text(text)
    #phones = speech.modulo1y2(clean_text, mode='Spell', PhTSimple='y', language=language, keep_chars=None, verbose=True)
    command = f"echo {cleaned_text} | iconv -f UTF-8 -t ISO-8859-1 | ./modulo1y2 -HDic=dict/eu_dic -Lang=eu -TxtMode=Spell -PhTSimple=y 2> /dev/null | iconv -f ISO-8859-1 -t UTF-8"
    phones = os.popen(command).read()
    # print('phones:', phones)

    command = f"echo {cleaned_text} | iconv -f UTF-8 -t ISO-8859-1 | ./modulo1y2 -HDic=dict/eu_dic -Lang=eu -TxtMode=Word -PhTSimple=n 2> /dev/null | iconv -f ISO-8859-1 -t UTF-8"
    checker = os.popen(command).read()
    # print('checker:', checker)
    #checker = speech.modulo1y2(clean_text, mode='Word', PhTSimple='n', language=language, keep_chars=None, verbose=False)
    # phones = phones.replace(" ", "")
    clp = ""
    for p in range(len(phones)):
        clp = clp + "".join(phones[p].split('-'))
    #     # print(f"clp: {clp}, phone: {phones[p]}")
    #     if p == len(phones) - 1:
    #         clp = clp + ' | '
    
    slp = str(clp).split()
    # print(slp)
    if '?' in checker:
        slp.append('?')
    elif '!' in checker:
        slp.append('!')
    elif '.' in checker:
        slp.append('.')
    else:
        slp.append('.')
    
    
    
    if checker[-2] == ':' or checker[-2] == ';':
        slp.append('.')
    phones = np.array(slp)

    return phones

def get_text(text, hps, language, path=False):
    if not path:
        text = getPhones(text, language)
        # print(f'text: {text}')
    text_norm = text_to_sequence(text, hps.data.text_cleaners, language, inference=not path)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

Usando speech desde: /home/asudupe/Latxa-Omni/dataset_generation/vits/speech.py


In [8]:
hps_marina = get_hparams_from_file("./configs/sonora.json")

net_g_marina = SynthesizerTrn(
    len(symbols),
    hps_marina.data.filter_length // 2 + 1,
    hps_marina.train.segment_size // hps_marina.data.hop_length,
    **hps_marina.model).cuda()
_ = net_g_marina.eval()

_ = load_checkpoint("./checkpoints/marina_898.pth", net_g_marina, None)

hps_aintzane = get_hparams_from_file("./configs/aintzane_eu_3.json")

net_g_aintzane = SynthesizerTrn(
    len(symbols),
    hps_aintzane.data.filter_length // 2 + 1,
    hps_aintzane.train.segment_size // hps_aintzane.data.hop_length,
    **hps_aintzane.model).cuda()
_ = net_g_aintzane.eval()

_ = load_checkpoint("./checkpoints/aintzane_3382.pth", net_g_aintzane, None)

hps_kiko = get_hparams_from_file("./configs/kiko_eu_2.json")

net_g_kiko = SynthesizerTrn(
    len(symbols),
    hps_kiko.data.filter_length // 2 + 1,
    hps_kiko.train.segment_size // hps_kiko.data.hop_length,
    **hps_kiko.model).cuda()
_ = net_g_kiko.eval()

_ = load_checkpoint("./checkpoints/kiko_4374.pth", net_g_kiko, None)

hps_alex = get_hparams_from_file("./configs/sonora.json")

net_g_alex = SynthesizerTrn(
    len(symbols),
    hps_alex.data.filter_length // 2 + 1,
    hps_alex.train.segment_size // hps_alex.data.hop_length,
    **hps_alex.model).cuda()
_ = net_g_alex.eval()

_ = load_checkpoint("./checkpoints/alex_864.pth", net_g_alex, None)

hps_kristof = get_hparams_from_file("./configs/kristof_eu.json")

net_g_kristof = SynthesizerTrn(
    len(symbols),
    hps_kristof.data.filter_length // 2 + 1,
    hps_kristof.train.segment_size // hps_kristof.data.hop_length,
    **hps_kristof.model).cuda()
_ = net_g_kristof.eval()

_ = load_checkpoint("./checkpoints/kristof_1697.pth", net_g_kristof, None)

/home/andoni.sudupe/.conda/envs/omni/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [50]:
# hps_nerea = get_hparams_from_file("/scratch/asudupe/models/vits/configs/nerea.json")

# net_g_nerea = SynthesizerTrn(
#     len(symbols),
#     hps_nerea.data.filter_length // 2 + 1,
#     hps_nerea.train.segment_size // hps_nerea.data.hop_length,
#     **hps_nerea.model)
# _ = net_g_nerea.eval()

# _ = load_checkpoint("/scratch/asudupe/models/vits/checkpoints/nerea_400000.pth", net_g_nerea, None)

hps_alex = get_hparams_from_file("/scratch/asudupe/models/vits/configs/sonora.json")

net_g_alex = SynthesizerTrn(
    len(symbols),
    hps_alex.data.filter_length // 2 + 1,
    hps_alex.train.segment_size // hps_alex.data.hop_length,
    **hps_alex.model)
_ = net_g_alex.eval()

_ = load_checkpoint("/scratch/asudupe/models/vits/checkpoints/alex_864.pth", net_g_alex, None)

hps_comb = get_hparams_from_file("/scratch/asudupe/models/vits/configs/multispeaker.json")

net_g_comb = SynthesizerTrn(
    len(symbols),
    hps_comb.data.filter_length // 2 + 1,
    hps_comb.train.segment_size // hps_comb.data.hop_length,
    n_speakers=9,
    **hps_comb.model)
_ = net_g_comb.eval()

_ = load_checkpoint("/scratch/asudupe/models/vits/checkpoints/multispeaker_500000.pth", net_g_comb, None)

/scratch/asudupe/conda-env/latxa-txat/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [58]:
def infer_voice(question, voice):

    net_g = net_g_comb
    hps = hps_comb
        
    net_g = net_g_alex
    hps = hps_alex
    stn_tst = get_text(question, hps, language='eu')
    with torch.no_grad():
        x_tst = stn_tst.unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)])
        sid = torch.LongTensor([12340])
        audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
        return audio

In [59]:
audio = infer_voice('Emango zenizkidake Adimen Artifizialaren bost adibide?', 1)
from IPython.display import Audio as show_audio

show_audio(audio, rate=22050)


In [48]:
from scipy.io.wavfile import write

In [49]:
write('test.wav', 22050, audio)

In [23]:
from datasets import Audio

In [ ]:
audio_audio = Audio(audio)

In [10]:
from tqdm import tqdm

In [221]:
# for ex in dataset.select(range(100)):
ex = dataset[160]
question = ex['question'][8:]
answer = ex['answer']

stn_tst = get_text(question, hps_marina, language='eu')
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g_marina.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
    write(f"{ex['id']}_user_{round}.wav", hps_marina.data.sampling_rate, audio)
stn_tst = get_text(answer, hps_marina, language='eu')
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g_marina.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
    write(f"{ex['id']}_assistant_{round}.wav", hps_marina.data.sampling_rate, audio)
# break

In [20]:
import multiprocess
# multiprocess.set_start_method("spawn")
multiprocess.set_start_method('forkserver', force=True)

In [ ]:
def infer_voice(question, voice): 
    if voice == 0:
        net_g = net_g_marina
        hps = hps_marina
    elif voice == 1:
        net_g = net_g_kiko
        hps = hps_kiko 
    elif voice == 2:
        net_g = net_g_alex
        hps = hps_alex
    elif voice == 3:
        net_g = net_g_kristof
        hps = hps_kristof
    elif voice == 4:
        net_g = net_g_aintzane
        hps = hps_aintzane 
    else: 
        print("Voice not recognized. Using default (marina).") 
        net_g = net_g_marina 
        hps = hps_marina 
    
    stn_tst = get_text(question, hps, language='eu')
    with torch.no_grad(): 
        x_tst = stn_tst.cuda().unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda() 
        audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy() 
        return audio 
def process(ex): 
    question = ex['question'][8:] 
    answer = ex['answer'] 
    random_voice = np.random.randint(0,5) 
    # print("Using voice:", random_voice)
    ex["question_audio"] = infer_voice(question, random_voice)
    ex["answer_audio"] = infer_voice(answer, 4)
dataset.map(process, num_proc=4)

NameError: name 'dataset' is not defined

In [246]:
def infer_voice_batch(texts, voice):
    if voice == 0:
        net_g = net_g_marina
        hps = hps_marina
    elif voice == 1:
        net_g = net_g_kiko
        hps = hps_kiko
    elif voice == 2:
        net_g = net_g_alex
        hps = hps_alex
    elif voice == 3:
        net_g = net_g_kristof
        hps = hps_kristof
    elif voice == 4:
        net_g = net_g_aintzane
        hps = hps_aintzane
    else:
        print("Voice not recognized. Using default (marina).")
        net_g = net_g_marina
        hps = hps_marina

    # Convert all texts in the batch
    stn_list = [get_text(txt, hps, language="eu") for txt in texts]

    # Pad to same length for batch inference
    max_len = max(s.size(0) for s in stn_list)
    x_tst = torch.zeros(len(stn_list), max_len, dtype=stn_list[0].dtype)
    x_tst_lengths = []
    for i, s in enumerate(stn_list):
        x_tst[i, : s.size(0)] = s
        x_tst_lengths.append(s.size(0))

    x_tst = x_tst.cuda()
    x_tst_lengths = torch.LongTensor(x_tst_lengths).cuda()

    with torch.no_grad():
        audio = net_g.infer(
            x_tst, x_tst_lengths,
            noise_scale=0.667,
            noise_scale_w=0.8,
            length_scale=1
        )[0]   # shape: (B, 1, T)

    audios = [a[0].cpu().numpy() for a in audio]  # list of numpy arrays
    return audios


In [ ]:
def process_batch(examples):
    # print(examples['question'])
    questions = [ex[8:] for ex in examples['question']]
    answers   = [ex for ex in examples["answer"]]

    # Random voices for each question
    question_voices = np.random.randint(0, 5)

    question_audios = infer_voice_batch(answers, question_voices)

    # Run batch inference for answers (always voice=4 here)
    answer_audios = infer_voice_batch(answers, 4)

    # Add back to examples
    # for i, ex in enumerate(examples):
    #     ex["question_audio"] = question_audios[i]
    #     ex["answer_audio"]   = answer_audios[i]

    return question_audios, answer_audios


In [305]:
question_audios[0]

array([ 3.64681473e-05, -1.09811706e-04, -2.52347727e-05, ...,
       -1.87536720e-02, -1.86097920e-02, -1.84375867e-02], dtype=float32)

In [288]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=32, shuffle=True)

all_question = []
all_answer = []
for batch in tqdm(loader):
    question_audios, answer_audios = process_batch(batch)
    all_question.extend(question_audios)
    all_answer.extend(answer_audios)


  0%|          | 5/10315 [00:14<8:25:29,  2.94s/it]


KeyboardInterrupt: 